# 04 · Dữ liệu SFT cho LoRA

Dựng dữ liệu **supervised fine-tuning** từ các lượt chạy có nhãn trên **train** (rejection sampling), rồi chia train / dev **theo shop**.

> Không bắt buộc: bộ SFT có sẵn trong `data/data/` (6.363 / 1.064 dòng, đã chia theo shop) là dữ liệu mặc định của `05_train_lora`.
> Notebook này là cách thứ hai, dựng dữ liệu từ lượt chạy của chính pipeline trên dữ liệu BTC (cần `data/train` và GPU cho bước 1).

| Bước | Script | Cần | Kết quả |
|---|---|---|---|
| 1. Lượt chạy có nhãn | `02_inference` với `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True`, `IDS = "splits/rft_train24.txt"` | GPU bf16 ≥ 24 GB | 8 mẫu thường + tối đa 4 mẫu thinking mỗi câu, kèm prompt đã gửi |
| 2. Chọn lời giải đúng | `training/build_sft.py` | CPU | `all.jsonl`: mỗi câu giữ lời giải đúng trên route tốt nhất (macro công nợ → view chuẩn → bảng thô); câu khó (0-5/8 đúng) tới 2 dòng, câu dễ góp `EASY` dòng, mẫu thinking đúng thành dòng thinking |
| 3. Ghi chú trước SQL | `training/add_notes.py` | GPU (vLLM) | `all_notes.jsonl`: dòng thường có vài dòng `Notes:` trước SQL, đúng định dạng `--notes` lúc suy luận |
| 4. Chia theo shop | `training/split_sft.py` | CPU | `train.jsonl`, `dev.jsonl`, `split_manifest.json` |
| 5. Kiểm tra chất lượng | notebook này | CPU | phân bố route / thinking / shop, độ dài, trùng lặp, định dạng trả lời |

**Vì sao chia theo shop:** các dòng của một shop dùng chung README, schema và thường cả mẫu SQL. Chia ngẫu nhiên theo dòng đặt các bản gần
giống nhau ở cả hai phía, dev loss khi đó không còn đo khả năng tổng quát sang shop mới (điều public / private đòi hỏi).

**Vì sao bỏ shop holdout:** 6 shop của `splits/holdout600.txt` không vào train cũng không vào dev, nên chạy pipeline trên holdout là phép đo sạch
cho adapter mới. LoRA v3 đã học 390 câu holdout nên điểm holdout của nó chỉ là smoke test (`docs/leaderboard.md`).

Chỉ dùng dữ liệu train: `split_sft.py` từ chối mọi dòng có id không phải câu train. Câu public / private không bao giờ vào dữ liệu train.

In [ ]:
# Setup: tìm checkout của repo này, nạp datamate.notebook, phát hiện dữ liệu, GPU, thư mục model và runs.
#   local   mở notebook trong notebooks/ của checkout (hoặc đặt biến môi trường DATAMATE_REPO), không cần cài thêm.
#   Kaggle  đính kèm bản sao checkout làm dataset (Add data) cùng dữ liệu BTC (dataset riêng tư), bật Internet để cài gói.
#   Colab   chép checkout vào MyDrive/datamate/code, dữ liệu BTC vào MyDrive/datamate/data.
# Kaggle / Colab: code được chép sang ổ ghi được (input của Kaggle chỉ đọc, Drive chậm); kết quả ghi vào env.runs, được giữ sau phiên.
import os, shutil, sys
from pathlib import Path

MARKER = "datamate/notebook.py"


def find_checkout() -> Path:
    if os.environ.get("DATAMATE_REPO"):
        return Path(os.environ["DATAMATE_REPO"]).resolve()
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / MARKER).exists():
            return c.resolve()
    if Path("/kaggle/input").exists():
        found, work = sorted(p.parents[1] for p in Path("/kaggle/input").glob(f"**/{MARKER}")), Path("/tmp/datamate")
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        found, work = [p for p in [Path("/content/drive/MyDrive/datamate/code")] if (p / MARKER).exists()], Path("/content/datamate")
    else:
        found, work = [], None
    if not found:
        raise FileNotFoundError("không tìm thấy checkout datamate: mở notebook trong notebooks/ của repo, hoặc đặt DATAMATE_REPO")
    if not (work / MARKER).exists():
        shutil.copytree(found[0], work, ignore=shutil.ignore_patterns(".git", "runs", "models", ".venv*", "__pycache__"))
    return work


root = find_checkout()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("llm")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
RUNS = ["runs/nb_train_sql_rft_v3"]   # lượt chạy có nhãn trên train với SAVE_PROMPTS (thư mục có trace.jsonl hoặc shard*/trace.jsonl)
TEACHER_RUNS = []                     # tùy chọn: lượt chạy của model lớn hơn trên các câu train mà 9B chưa giải được
OUT_DIR = "runs/sft/sql_rft_nb"      # all.jsonl, all_notes.jsonl, train.jsonl, dev.jsonl, split_manifest.json
EASY = 800                            # số dòng lấy từ câu dễ (6-8 trên 8 mẫu đúng)
MAX = 4000                            # trần số dòng (dòng công nợ và teacher luôn được giữ)
ADD_NOTES = True                      # cần GPU bf16 >= 24 GB (vLLM, trọng số gốc)
DEV_SHOPS = 2                         # số shop làm dev, hoặc danh sách (ví dụ ["biz_104", "biz_119"]); 0 = fit cuối trên mọi shop
EXCLUDE_HOLDOUT = True                # bỏ dòng của 6 shop holdout (splits/holdout_shops.txt) khỏi cả train và dev
SEED = 0                              # chọn shop dev
BUILD_BANK = False                    # ghi thêm ngân hàng few-shot từ RUNS

In [ ]:
def trace_dir(r):
    """build_sft.py đọc RUN/trace.jsonl: lượt chạy nhiều GPU được gộp shard trước."""
    r = env.path(r)
    if not (r / "trace.jsonl").exists() and list(r.glob("shard*/trace.jsonl")):
        with (r / "trace.jsonl").open("w", encoding="utf-8") as f:
            for p in sorted(r.glob("shard*/trace.jsonl")):
                f.write(p.read_text(encoding="utf-8"))
    if not (r / "trace.jsonl").exists():
        raise FileNotFoundError(f"{r}: không có trace.jsonl. Chạy 02_inference với SPLIT = 'train', SAVE_PROMPTS = True trước")
    return r


gold = {a["id"]: a for a in read(env.split("train") / "answers.jsonl")}
q_shop = {q["id"]: q["business_id"] for q in read(env.data / "train/questions.jsonl")}
holdout_shops = set((env.root / "splits/holdout_shops.txt").read_text().split())
runs, teachers = [trace_dir(r) for r in RUNS], [trace_dir(r) for r in TEACHER_RUNS]
summary = []
for r in runs + teachers:
    t = read(r / "trace.jsonl")
    summary.append({"run": r.name, "questions": len(t), "train ids": sum(x["id"] in gold for x in t),
                    "saved prompts": sum(bool(x.get("prompt_user")) for x in t), "holdout-shop questions": sum(q_shop.get(x["id"]) in holdout_shops for x in t),
                    "samples / question": round(sum(len(x["samples"]) for x in t) / max(len(t), 1), 1)})
summary = pd.DataFrame(summary)
display(summary)
assert (summary["train ids"] == summary["questions"]).all(), "chỉ dùng lượt chạy trên train (có đáp án)"
assert (summary["saved prompts"] > 0).all(), "lượt chạy thiếu prompt: chạy lại 02_inference với SAVE_PROMPTS = True"

In [ ]:
out_dir = env.path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
all_rows = out_dir / "all.jsonl"
env.run([env.python, "training/build_sft.py", *runs, *(["--teacher", *teachers] if teachers else []), "--fewshot", "assets/fewshot_bank.jsonl",
         "--out", all_rows, "--max", MAX, "--easy", EASY], log=out_dir / "build_sft.log", show="available|rows by|Traceback|Error")

### Ghi chú trước SQL

Lúc suy luận, mẫu thường viết vài dòng `Notes:` (khung thời gian, bộ lọc, đơn vị) trước câu SQL (`--notes`). Dòng SFT cần đúng định dạng đó,
nếu không adapter học một định dạng khác với lúc dùng. `add_notes.py` sinh ghi chú bằng model gốc (greedy) và giữ nguyên dòng khi câu trả lời
sai định dạng.

In [ ]:
sft_all = all_rows
if ADD_NOTES and env.can_run_llm:
    flags, _ = env.inference_flags()                      # Ampere: bf16, Ada / Hopper / Blackwell: fp8 như lúc suy luận
    sft_all = out_dir / "all_notes.jsonl"
    env.run([env.python, "training/add_notes.py", all_rows, sft_all, "--model", env.base_model(), "--rulebook", env.data / "train/rulebook.md",
             "--quant", "fp8" if "--quant" in flags else "none"], log=out_dir / "add_notes.log", show="notes on|Traceback|Error")
elif ADD_NOTES:
    try:                                                  # không có GPU: tệp ghi chú đã chạy sẵn trên GPU 24 GB (runs/saved), nếu có
        sft_all = env.use_saved("all_notes.jsonl", out_dir / "all_notes.jsonl")
    except RuntimeError as e:
        print("bỏ qua bước ghi chú:", e, "\n=> dòng thường không có 'Notes:', khác định dạng --notes lúc suy luận")
print("dữ liệu SFT trước khi chia:", sft_all, "|", sum(1 for _ in open(sft_all, encoding="utf-8")), "dòng")

### Chia train / dev theo shop

`DEV_SHOPS` shop (chọn ngẫu nhiên với `SEED` trong nhóm shop có số dòng ở khoảng giữa) làm dev: dev loss trong lúc train và chọn checkpoint
tốt nhất ở `05_train_lora`. Sau khi đã chọn số bước, có thể train lại trên mọi shop: `DEV_SHOPS = 0` ở đây, `DEV = None` và
`MAX_STEPS` = số bước của checkpoint tốt nhất ở `05_train_lora`.

In [ ]:
dev_arg = ",".join(DEV_SHOPS) if isinstance(DEV_SHOPS, (list, tuple)) else str(DEV_SHOPS)
env.run([env.python, "training/split_sft.py", sft_all, "--out-dir", out_dir, "--dev-shops", dev_arg, "--seed", SEED,
         "--holdout-shops", "splits/holdout_shops.txt" if EXCLUDE_HOLDOUT else ""], show="rows in|split ->|Traceback|Error")
manifest = json.loads((out_dir / "split_manifest.json").read_text(encoding="utf-8"))
pd.DataFrame({s: {"rows": manifest[s]["rows"], "questions": manifest[s]["questions"], "thinking rows": manifest[s]["thinking_rows"],
                  "shops": len(manifest[s]["shops"])} for s in ("train", "dev")}).T

### Kiểm tra chất lượng dữ liệu

- **Route:** `macro` (macro công nợ), `view` (view chuẩn `v_sales` / `v_orders`), `raw` (bảng thô). Lời giải trên view / macro ngắn và ít sai hơn.
- **Thinking:** dòng thinking dạy chế độ suy nghĩ (dùng cho câu khó). Tỷ lệ quá cao làm adapter thiên về câu trả lời dài.
- **Độ dài:** prompt chiếm phần lớn mỗi dòng (rulebook + schema), loss chỉ tính trên phần trả lời.
- **Định dạng:** câu trả lời phải có khối ```` ```sql ````; dòng thường nên bắt đầu bằng `Notes:`. `05_train_lora` kiểm tra lại bằng tokenizer thật.

In [ ]:
import hashlib

data = {s: read(out_dir / f"{s}.jsonl") for s in ("train", "dev")}
df = pd.DataFrame([{"split": s, "id": r["id"], "shop": q_shop[r["id"]], "route": r.get("route"), "thinking": bool(r.get("thinking")),
                    "prompt_chars": len(r["system"]) + len(r["user"]), "reply_chars": len(r["reply"]),
                    "notes": r["reply"].startswith("Notes:"), "sql_block": "```sql" in r["reply"].split("</think>")[-1],
                    "key": hashlib.sha1("\0".join((r["system"], r["user"], r["reply"])).encode()).hexdigest()}
                   for s, rows in data.items() for r in rows])
display(df.groupby(["split", "route", "thinking"]).size().unstack(fill_value=0))
display(df.groupby("split")[["prompt_chars", "reply_chars"]].describe().round(0))
plain = df[~df.thinking]
print(f"dòng thường có Notes: {plain.notes.mean():.1%} | có khối ```sql: {df.sql_block.mean():.1%} | "
      f"dòng trùng lặp: {df.duplicated('key').sum()} | câu hỏi có > 1 dòng: {(df.groupby('id').size() > 1).sum()}")
assert not set(df[df.split == "train"].shop) & set(df[df.split == "dev"].shop), "train và dev chung shop"
assert not set(df.shop) & holdout_shops or not EXCLUDE_HOLDOUT, "còn dòng của shop holdout"
df.groupby(["shop", "split"]).size().unstack(fill_value=0).plot.barh(stacked=True, figsize=(7, 6), title="rows per shop");

In [ ]:
r = data["train"][0]
print(r["user"][-600:], "\n---- reply ----\n", r["reply"][:800])

In [ ]:
if BUILD_BANK:
    env.run([env.python, "training/build_fewshot_bank.py", *runs, "--out", out_dir / "fewshot_bank.jsonl"], show="Traceback|Error|bank|rows")
print("05_train_lora: TRAIN =", repr(str(out_dir / "train.jsonl")), "| DEV =", repr(str(out_dir / "dev.jsonl")) if manifest["dev"]["rows"] else None)